<div align='center'>

# 🤖 KRONOS RL — Deep Reinforcement Learning
## *Orbit Wars · $50,000 Prize · Deadline June 16, 2026*

</div>

---

## Architecture

```
Game Environment (Gymnasium)
        ↓  observation (60-dim)
   [PPO Policy Network: 256→256→128]
        ↓  action (src, tgt, fraction)
   orbit_wars step()
        ↓  reward (shaped)
   Backprop → update weights
```

## Reward Shaping:

| Signal | Value | Purpose |
|--------|-------|---------|
| Production gained | +1.0 per prod unit | grow economy |
| Planet captured | +10 | expand |
| Planet lost | -15 | protect |
| Ship dominance | +0.5×(ratio-0.25) | maintain army |
| Early expansion bonus | +0.5×prod (step<100) | open fast |
| Win | +100 | ultimate goal |
| Loss | -50 | avoid defeat |

---


## ⚙️ Cell 1 — Install


In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'
!pip install gymnasium stable-baselines3 torch --quiet
print('✅ All packages installed')


## 📦 Cell 2 — Imports


In [ ]:
import math, time, random, collections
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from kaggle_environments import make

env_check = make('orbit_wars', debug=True)
env_check.reset()
obs_raw = dict(env_check.state[0].observation)
print(f'✅ orbit_wars v{env_check.version} | planets={len(obs_raw["planets"])} | av={obs_raw["angular_velocity"]:.4f}')


## ⚙️ Cell 3 — Physics Core


In [ ]:
SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def copy(self):
        p=_P(); [setattr(p,f,getattr(self,f)) for f in self.__slots__]; return p

class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

def spd(n):  return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):  return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=18):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.02: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=32):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False
def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(14):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf): hi=mid
        else: lo=mid+1
    return hi



## 🔬 Cell 4 — KRONOS Heuristic (Opponent + Rollout)


In [ ]:
def kronos_agent(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine=[p for p in planets if p.owner==pl]
    others=[p for p in planets if p.owner!=pl]
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()

    def gn(p): return max(3, int(p.ships*0.10), p.production*2)
    def sp(p): return p.ships-used.get(p.id,0)-gn(p)

    # Incoming threats → defense
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_=icp(f.x,f.y,p,av,f.ships)
            if dd<p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=int(thr*1.12)+5; deficit=need-p.ships+used.get(p.id,0)
        if deficit<=0: continue
        for src in sorted([s for s in mine if s.id!=p.id and sp(s)>4],
                          key=lambda s:d2(s.x,s.y,p.x,p.y))[:2]:
            snd=min(sp(src),deficit)
            if snd<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,snd); sa,ok=safe(src.x,src.y,a,dd)
            if ok: moves.append([src.id,sa,snd]); used[src.id]=used.get(src.id,0)+snd; deficit-=snd
            if deficit<=0: break

    # En-route
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<80: enroute.add(t.id)

    # Score and attack
    cands=[]
    for src in mine:
        if sp(src)<3: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,eta=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            tw=max(0,rem-eta); prod=tgt.production
            sc=(prod**2)*12*tw+prod*tw
            if tgt.owner>=0: sc*=1.6
            if tgt.ships<=prod*2+2: sc*=2.0
            sc-=n*0.35
            cands.append((sc,id(src),src,tgt,n,sa))

    cands.sort(key=lambda x:-x[0])
    max_atk=5 if stp<80 else 4
    atks=0
    for sc,_,src,tgt,n,sa in cands:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if sp(src)<n: continue
        moves.append([src.id,sa,n]); used[src.id]=used.get(src.id,0)+n
        done.add(tgt.id); atks+=1

    # Sweep
    for src in sorted(mine,key=lambda p:-sp(p)):
        if sp(src)<4: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            sc2=tgt.production*10/(dd+1)+(1.6 if tgt.owner>=0 else 1.0)
            if sc2>bsc: bsc=sc2; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            used[best[0]]=used.get(best[0],0)+best[2]; done.add(best[3])

    return moves



## 🌍 Cell 5 — OrbitWarsEnv (Complete Gymnasium Environment)

**Observation:** 60-dim flat vector (20 planets × 3 features)

**Action:** `MultiDiscrete([20, 20, 10])`
- `src_idx`: which of our planets to send from
- `tgt_idx`: which enemy/neutral planet to attack
- `fraction`: 10%–100% of ships to send

**Reward:** shaped per-tick + terminal


In [ ]:
class OrbitWarsEnv(gym.Env):
    """
    Full Gymnasium environment for Orbit Wars.
    
    Observation (60-dim flat vector):
      Per planet (max 20): [owner_norm, ships_norm, prod_norm] = 60 dims
    
    Action: MultiDiscrete([20, 20, 10])
      - src planet index (0-19)
      - tgt planet index (0-19)  
      - ships fraction (0=10%, 1=20%, ..., 9=100%)
    
    Reward Shaping (per tick + terminal):
      +prod_gained * 0.5    gaining production
      +ship_ratio * 0.3     having more ships than enemy
      +10 per planet captured
      -15 per planet lost
      -0.05 * ships_wasted  wasting ships on sun/border
      +100 win / -50 loss
    """
    metadata = {'render_modes': []}

    MAX_PLANETS = 20

    def __init__(self, player_id=0, opponent=None):
        super().__init__()
        self.player_id = player_id
        self.opponent  = opponent or kronos_agent

        # ── Observation space: 20 planets × 3 features ──────────────────
        # features: [owner_norm, ships_norm, production_norm]
        # owner: -1=neutral(0), 0=us(0.5), other=(owner+1)/4
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0,
            shape=(self.MAX_PLANETS * 3,),
            dtype=np.float32
        )

        # ── Action space ─────────────────────────────────────────────────
        # src_idx (0-19), tgt_idx (0-19), fraction (0-9 → 10%-100%)
        self.action_space = spaces.MultiDiscrete([20, 20, 10])

        # Internal state
        self._env          = None
        self._prev_my_prod = 0
        self._prev_my_cnt  = 0
        self._prev_ships   = 0
        self._step         = 0

    def _make_kaggle_env(self):
        self._env = make('orbit_wars', debug=False)

    def _parse_obs(self):
        """Parse kaggle observation into our format."""
        raw = dict(self._env.state[0].observation)
        planets = [_P(*p) for p in raw.get('planets', [])]
        fleets  = [_F(*f) for f in raw.get('fleets',  [])]
        av      = raw.get('angular_velocity', 0.0366)
        stp     = raw.get('step', 0)
        return planets, fleets, av, stp, raw

    def _build_obs_vector(self, planets):
        """Build flat observation vector from planet list."""
        obs = np.zeros((self.MAX_PLANETS, 3), dtype=np.float32)
        for i, p in enumerate(planets[:self.MAX_PLANETS]):
            # owner_norm: neutral=0, us=1, enemy=-1
            if p.owner < 0:         obs[i,0] = 0.0
            elif p.owner==self.player_id: obs[i,0] = 1.0
            else:                   obs[i,0] = -1.0
            obs[i,1] = min(1.0, p.ships / 200.0)
            obs[i,2] = p.production / 10.0
        return obs.flatten()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._make_kaggle_env()
        self._env.reset()
        planets,_,_,stp,_ = self._parse_obs()
        self._prev_my_prod = sum(p.production for p in planets if p.owner==self.player_id)
        self._prev_my_cnt  = sum(1 for p in planets if p.owner==self.player_id)
        self._prev_ships   = sum(p.ships for p in planets if p.owner==self.player_id)
        self._step = 0
        return self._build_obs_vector(planets), {}

    def step(self, action):
        planets, fleets, av, stp, raw = self._parse_obs()

        mine   = [p for p in planets if p.owner==self.player_id]
        others = [p for p in planets if p.owner!=self.player_id]

        # ── Decode action ─────────────────────────────────────────────
        src_idx  = int(action[0])
        tgt_idx  = int(action[1])
        frac_idx = int(action[2])  # 0-9 → 10%-100%
        fraction = (frac_idx + 1) / 10.0

        moves = []
        if src_idx < len(mine) and tgt_idx < len(others):
            src = mine[src_idx]
            tgt = others[tgt_idx]
            n   = max(1, int(src.ships * fraction))
            n   = min(n, src.ships - 1)

            if n > 0:
                a, dd, _ = icp(src.x, src.y, tgt, av, n)
                sa, ok   = safe(src.x, src.y, a, dd)
                if ok:
                    moves.append([src.id, sa, n])

        # ── Step the kaggle environment ───────────────────────────────
        # Build actions for all 4 players
        all_moves = [None, None, None, None]
        all_moves[self.player_id] = moves
        for pid in range(4):
            if pid != self.player_id:
                all_moves[pid] = kronos_agent(
                    self._env.state[pid].observation
                )
        self._env.step(all_moves)
        self._step += 1

        # ── Get new state ─────────────────────────────────────────────
        new_planets,_,_,new_stp,_ = self._parse_obs()
        obs = self._build_obs_vector(new_planets)

        # ── REWARD SHAPING ────────────────────────────────────────────
        reward = 0.0

        my_prod = sum(p.production for p in new_planets if p.owner==self.player_id)
        my_cnt  = sum(1 for p in new_planets if p.owner==self.player_id)
        my_ships= sum(p.ships for p in new_planets if p.owner==self.player_id)
        all_ships=max(1,sum(p.ships for p in new_planets if p.owner>=0))
        enemy_ships=max(1,sum(p.ships for p in new_planets if p.owner>=0 and p.owner!=self.player_id))

        # 1. Production gained (per tick)
        prod_delta = my_prod - self._prev_my_prod
        reward += prod_delta * 1.0

        # 2. Planet captured/lost
        cnt_delta = my_cnt - self._prev_my_cnt
        if cnt_delta > 0: reward += cnt_delta * 10.0  # captured
        if cnt_delta < 0: reward += cnt_delta * 15.0  # lost (negative)

        # 3. Ship dominance ratio (per tick)
        ship_ratio = my_ships / all_ships
        reward += (ship_ratio - 0.25) * 0.5  # bonus for >25% of total ships

        # 4. Efficiency: reward for expanding early
        if new_stp < 100 and prod_delta > 0:
            reward += prod_delta * 0.5  # double bonus in early game

        # 5. Penalty for idle ships (ships not growing = wasted turns)
        if my_ships < self._prev_ships and cnt_delta >= 0:
            reward -= 0.02  # small penalty for losing ships without gaining planets

        self._prev_my_prod = my_prod
        self._prev_my_cnt  = my_cnt
        self._prev_ships   = my_ships

        # ── Terminal reward ───────────────────────────────────────────
        status = self._env.state[self.player_id].status
        done   = (status != 'ACTIVE')

        if done:
            r = self._env.state[self.player_id].reward
            if   r ==  1: reward += 100.0  # WIN
            elif r == -1: reward -= 50.0   # LOSS
            else:         reward -= 10.0   # DRAW

        truncated = (new_stp >= MS)
        return obs, reward, done, truncated, {}

    def render(self): pass
    def close(self):
        if self._env: self._env = None



## 🧪 Cell 6 — Environment Sanity Test


In [ ]:
# Test the environment works correctly
test_env = OrbitWarsEnv(player_id=0)
obs, info = test_env.reset()
print(f'✅ Env reset OK')
print(f'   Obs shape    : {obs.shape}')
print(f'   Obs dtype    : {obs.dtype}')
print(f'   Action space : {test_env.action_space}')
print(f'   Obs space    : {test_env.observation_space}')

# Test a few random steps
total_reward = 0
for i in range(5):
    action = test_env.action_space.sample()
    obs, reward, done, trunc, info = test_env.step(action)
    total_reward += reward
    if done: break

print(f'   5-step reward: {total_reward:.2f}')
print(f'✅ Environment working correctly!')
test_env.close()


## 🚀 Cell 7 — PPO Training

> ⏰ 100k steps ≈ 30 min on CPU, 10 min on GPU
> ⏰ 500k steps ≈ 2-3 hours (recommended)
> ⏰ 2M steps  ≈ overnight (best results)


In [ ]:
def train_ppo(total_timesteps=500_000, save_path='kronos_ppo'):
    """
    Train PPO agent with reward-shaped OrbitWarsEnv.
    Self-play: agent trains against KRONOS heuristic.
    """
    try:
        from stable_baselines3 import PPO
        from stable_baselines3.common.callbacks import (
            EvalCallback, CheckpointCallback
        )
        from stable_baselines3.common.monitor import Monitor
        import torch

        # Create environments
        train_env = Monitor(OrbitWarsEnv(player_id=0))

        # PPO configuration
        model = PPO(
            'MlpPolicy',
            train_env,
            verbose=1,
            learning_rate=3e-4,
            n_steps=256,
            batch_size=64,
            n_epochs=8,
            gamma=0.995,        # values long-term production
            gae_lambda=0.95,
            clip_range=0.2,
            ent_coef=0.01,      # encourages exploration
            policy_kwargs=dict(
                net_arch=dict(
                    pi=[256, 256, 128],  # policy network
                    vf=[256, 256, 128]   # value network
                ),
                activation_fn=torch.nn.ReLU
            )
        )

        print("🤖 PPO Model architecture:")
        print(f"   Obs dims : {train_env.observation_space.shape}")
        print(f"   Act dims : {train_env.action_space.nvec}")
        print(f"   Network  : [256, 256, 128]")
        print(f"   Params   : {sum(p.numel() for p in model.policy.parameters()):,}")
        print(f"\n🚀 Training for {total_timesteps:,} steps...")

        # Checkpoint every 50k steps
        checkpoint_cb = CheckpointCallback(
            save_freq=50_000,
            save_path='./',
            name_prefix='kronos_checkpoint'
        )

        model.learn(
            total_timesteps=total_timesteps,
            callback=checkpoint_cb,
            progress_bar=True
        )

        model.save(save_path)
        print(f"\n✅ Model saved: {save_path}.zip")
        return model

    except ImportError as e:
        print(f"⚠️  {e}")
        print("   Run: !pip install stable-baselines3 torch")
        return None


# ── START TRAINING ────────────────────────────────────────────────────────
# Adjust total_timesteps based on available time:
#   Quick test   : 10_000
#   Competition  : 500_000 to 2_000_000

model = train_ppo(
    total_timesteps = 500_000,   # ← change this
    save_path       = 'kronos_ppo'
)


## 🤖 Cell 8 — Load PPO Agent & Test


In [ ]:
def make_ppo_agent(model_path='kronos_ppo'):
    """
    Load trained PPO model and return submission agent function.
    Falls back to KRONOS heuristic if model not found.
    """
    try:
        from stable_baselines3 import PPO
        model = PPO.load(model_path)
        print(f"✅ PPO model loaded from {model_path}")

        def ppo_orbital_strategist(obs):
            if isinstance(obs,dict):
                pl=obs.get('player',0); rp=obs.get('planets',[])
                rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
                stp=obs.get('step',0)
            else:
                pl=obs.player; rp=obs.planets; rf=obs.fleets
                av=obs.angular_velocity; stp=getattr(obs,'step',0)

            try:
                from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
                planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
            except:
                planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

            mine  =[p for p in planets if p.owner==pl]
            others=[p for p in planets if p.owner!=pl]

            # Build observation
            obs_arr = np.zeros((20,3), dtype=np.float32)
            for i,p in enumerate(planets[:20]):
                obs_arr[i,0] = 1.0 if p.owner==pl else (0.0 if p.owner<0 else -1.0)
                obs_arr[i,1] = min(1.0,p.ships/200.0)
                obs_arr[i,2] = p.production/10.0
            obs_flat = obs_arr.flatten().reshape(1,-1)

            # Predict action
            action,_ = model.predict(obs_flat, deterministic=True)
            src_idx,tgt_idx,frac_idx = int(action[0]),int(action[1]),int(action[2])

            if src_idx>=len(mine) or tgt_idx>=len(others): return []
            src=mine[src_idx]; tgt=others[tgt_idx]
            fraction=(frac_idx+1)/10.0
            n=max(1,int(src.ships*fraction)); n=min(n,src.ships-1)
            if n<=0: return []
            a,dd,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            return [[src.id,sa,n]] if ok else []

        return ppo_orbital_strategist

    except Exception as e:
        print(f"⚠️  PPO model not found ({e}), using KRONOS heuristic")
        return kronos_agent

# ─── SUBMISSION AGENT ────────────────────────────────────────────────────────
# Before training: uses KRONOS heuristic
# After training:  use make_ppo_agent() for PPO inference
orbital_strategist = kronos_agent
agent = orbital_strategist

# Load trained model and test against v1 baseline
from kaggle_environments import make

ppo_agent = make_ppo_agent('kronos_ppo')

def v1_agent(obs):
    return kronos_agent(obs)  # kronos IS our v1 equivalent

env_test = make('orbit_wars', debug=False)
env_test.run([ppo_agent, v1_agent, 'random', v1_agent])
rewards = [s.reward for s in env_test.steps[-1]]
labels = ['🤖 PPO', 'KRONOS', '🎲 Rand', 'KRONOS']
for lb, rw in zip(labels, rewards):
    print(f'  {"🏆" if rw==1 else "  "} {lb:10s} {rw:+d}')


## 📊 Cell 9 — Tournament 20 Games


In [ ]:
import random as _rnd

# Choose agent: ppo_agent (if trained) or kronos_agent
our_agent = ppo_agent  # switch to kronos_agent if PPO not ready

N=20; wins={'ours':0,'kronos':0,'rnd':0}
for g in range(N):
    agents=[our_agent, v1_agent, 'random', v1_agent]
    _rnd.shuffle(agents); kp=agents.index(our_agent)
    et=make('orbit_wars',debug=False); et.run(agents)
    rws=[s.reward for s in et.steps[-1]]; w=rws.index(max(rws))
    if w==kp: wins['ours']+=1; wl='🏆 Ours'
    elif agents[w]==v1_agent: wins['kronos']+=1; wl='KRONOS'
    else: wins['rnd']+=1; wl='🎲'
    print(f'G{g+1:02d}[A@{kp}] {[f"{r:+d}" for r in rws]} → {wl}')
print('─'*48)
for nm,w in wins.items(): print(f'  {nm:7s}: {w}/{N}  {"█"*(w*2)}')
wr=wins['ours']/N; elo=int(600+max(0,wr-0.25)*3800)
print(f'\n  Win rate : {wr:.0%}  |  Elo est: ~{elo}')
print(f'  {"🏆 MEDAL ZONE!" if elo>=1400 else "✅ Competitive" if elo>=1000 else "⚠️ Keep training"}')


## 💾 Cell 10 — Write Submission

> Before training: submits KRONOS heuristic
> After training: switch `orbital_strategist = ppo_agent`


In [ ]:
%%writefile main.py
# KRONOS RL Submission
# Switch to PPO agent after training
"""
KRONOS RL — Complete PPO Training System for Orbit Wars
$50,000 Competition — Deadline June 16, 2026

Reward Shaping Strategy:
  +production_gained    every tick we gain a planet
  +ship_ratio_lead      every tick our ships > enemy ships  
  +10 per planet captured
  -15 per planet lost
  +100 win / -50 loss
  -0.1 per ship wasted (fleet destroyed by sun)
"""

# ─── CELL 1: Install ─────────────────────────────────────────────────────────
# %%capture
# !pip install --upgrade "kaggle-environments>=1.28.0"
# !pip install gymnasium stable-baselines3 torch --quiet

# ─── CELL 2: Imports ─────────────────────────────────────────────────────────
import math, time, random, collections
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from kaggle_environments import make

# ─── CELL 3: Physics (shared) ────────────────────────────────────────────────
SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def copy(self):
        p=_P(); [setattr(p,f,getattr(self,f)) for f in self.__slots__]; return p

class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

def spd(n):  return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):  return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=18):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.02: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=32):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False
def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(14):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf): hi=mid
        else: lo=mid+1
    return hi

# ─── CELL 4: KRONOS Heuristic (used as opponent + rollout) ───────────────────
def kronos_agent(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine=[p for p in planets if p.owner==pl]
    others=[p for p in planets if p.owner!=pl]
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()

    def gn(p): return max(3, int(p.ships*0.10), p.production*2)
    def sp(p): return p.ships-used.get(p.id,0)-gn(p)

    # Incoming threats → defense
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_=icp(f.x,f.y,p,av,f.ships)
            if dd<p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=int(thr*1.12)+5; deficit=need-p.ships+used.get(p.id,0)
        if deficit<=0: continue
        for src in sorted([s for s in mine if s.id!=p.id and sp(s)>4],
                          key=lambda s:d2(s.x,s.y,p.x,p.y))[:2]:
            snd=min(sp(src),deficit)
            if snd<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,snd); sa,ok=safe(src.x,src.y,a,dd)
            if ok: moves.append([src.id,sa,snd]); used[src.id]=used.get(src.id,0)+snd; deficit-=snd
            if deficit<=0: break

    # En-route
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<80: enroute.add(t.id)

    # Score and attack
    cands=[]
    for src in mine:
        if sp(src)<3: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,eta=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            tw=max(0,rem-eta); prod=tgt.production
            sc=(prod**2)*12*tw+prod*tw
            if tgt.owner>=0: sc*=1.6
            if tgt.ships<=prod*2+2: sc*=2.0
            sc-=n*0.35
            cands.append((sc,id(src),src,tgt,n,sa))

    cands.sort(key=lambda x:-x[0])
    max_atk=5 if stp<80 else 4
    atks=0
    for sc,_,src,tgt,n,sa in cands:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if sp(src)<n: continue
        moves.append([src.id,sa,n]); used[src.id]=used.get(src.id,0)+n
        done.add(tgt.id); atks+=1

    # Sweep
    for src in sorted(mine,key=lambda p:-sp(p)):
        if sp(src)<4: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            sc2=tgt.production*10/(dd+1)+(1.6 if tgt.owner>=0 else 1.0)
            if sc2>bsc: bsc=sc2; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            used[best[0]]=used.get(best[0],0)+best[2]; done.add(best[3])

    return moves

# ─── CELL 5: COMPLETE GYMNASIUM ENVIRONMENT ──────────────────────────────────
class OrbitWarsEnv(gym.Env):
    """
    Full Gymnasium environment for Orbit Wars.
    
    Observation (60-dim flat vector):
      Per planet (max 20): [owner_norm, ships_norm, prod_norm] = 60 dims
    
    Action: MultiDiscrete([20, 20, 10])
      - src planet index (0-19)
      - tgt planet index (0-19)  
      - ships fraction (0=10%, 1=20%, ..., 9=100%)
    
    Reward Shaping (per tick + terminal):
      +prod_gained * 0.5    gaining production
      +ship_ratio * 0.3     having more ships than enemy
      +10 per planet captured
      -15 per planet lost
      -0.05 * ships_wasted  wasting ships on sun/border
      +100 win / -50 loss
    """
    metadata = {'render_modes': []}

    MAX_PLANETS = 20

    def __init__(self, player_id=0, opponent=None):
        super().__init__()
        self.player_id = player_id
        self.opponent  = opponent or kronos_agent

        # ── Observation space: 20 planets × 3 features ──────────────────
        # features: [owner_norm, ships_norm, production_norm]
        # owner: -1=neutral(0), 0=us(0.5), other=(owner+1)/4
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0,
            shape=(self.MAX_PLANETS * 3,),
            dtype=np.float32
        )

        # ── Action space ─────────────────────────────────────────────────
        # src_idx (0-19), tgt_idx (0-19), fraction (0-9 → 10%-100%)
        self.action_space = spaces.MultiDiscrete([20, 20, 10])

        # Internal state
        self._env          = None
        self._prev_my_prod = 0
        self._prev_my_cnt  = 0
        self._prev_ships   = 0
        self._step         = 0

    def _make_kaggle_env(self):
        self._env = make('orbit_wars', debug=False)

    def _parse_obs(self):
        """Parse kaggle observation into our format."""
        raw = dict(self._env.state[0].observation)
        planets = [_P(*p) for p in raw.get('planets', [])]
        fleets  = [_F(*f) for f in raw.get('fleets',  [])]
        av      = raw.get('angular_velocity', 0.0366)
        stp     = raw.get('step', 0)
        return planets, fleets, av, stp, raw

    def _build_obs_vector(self, planets):
        """Build flat observation vector from planet list."""
        obs = np.zeros((self.MAX_PLANETS, 3), dtype=np.float32)
        for i, p in enumerate(planets[:self.MAX_PLANETS]):
            # owner_norm: neutral=0, us=1, enemy=-1
            if p.owner < 0:         obs[i,0] = 0.0
            elif p.owner==self.player_id: obs[i,0] = 1.0
            else:                   obs[i,0] = -1.0
            obs[i,1] = min(1.0, p.ships / 200.0)
            obs[i,2] = p.production / 10.0
        return obs.flatten()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._make_kaggle_env()
        self._env.reset()
        planets,_,_,stp,_ = self._parse_obs()
        self._prev_my_prod = sum(p.production for p in planets if p.owner==self.player_id)
        self._prev_my_cnt  = sum(1 for p in planets if p.owner==self.player_id)
        self._prev_ships   = sum(p.ships for p in planets if p.owner==self.player_id)
        self._step = 0
        return self._build_obs_vector(planets), {}

    def step(self, action):
        planets, fleets, av, stp, raw = self._parse_obs()

        mine   = [p for p in planets if p.owner==self.player_id]
        others = [p for p in planets if p.owner!=self.player_id]

        # ── Decode action ─────────────────────────────────────────────
        src_idx  = int(action[0])
        tgt_idx  = int(action[1])
        frac_idx = int(action[2])  # 0-9 → 10%-100%
        fraction = (frac_idx + 1) / 10.0

        moves = []
        if src_idx < len(mine) and tgt_idx < len(others):
            src = mine[src_idx]
            tgt = others[tgt_idx]
            n   = max(1, int(src.ships * fraction))
            n   = min(n, src.ships - 1)

            if n > 0:
                a, dd, _ = icp(src.x, src.y, tgt, av, n)
                sa, ok   = safe(src.x, src.y, a, dd)
                if ok:
                    moves.append([src.id, sa, n])

        # ── Step the kaggle environment ───────────────────────────────
        # Build actions for all 4 players
        all_moves = [None, None, None, None]
        all_moves[self.player_id] = moves
        for pid in range(4):
            if pid != self.player_id:
                all_moves[pid] = kronos_agent(
                    self._env.state[pid].observation
                )
        self._env.step(all_moves)
        self._step += 1

        # ── Get new state ─────────────────────────────────────────────
        new_planets,_,_,new_stp,_ = self._parse_obs()
        obs = self._build_obs_vector(new_planets)

        # ── REWARD SHAPING ────────────────────────────────────────────
        reward = 0.0

        my_prod = sum(p.production for p in new_planets if p.owner==self.player_id)
        my_cnt  = sum(1 for p in new_planets if p.owner==self.player_id)
        my_ships= sum(p.ships for p in new_planets if p.owner==self.player_id)
        all_ships=max(1,sum(p.ships for p in new_planets if p.owner>=0))
        enemy_ships=max(1,sum(p.ships for p in new_planets if p.owner>=0 and p.owner!=self.player_id))

        # 1. Production gained (per tick)
        prod_delta = my_prod - self._prev_my_prod
        reward += prod_delta * 1.0

        # 2. Planet captured/lost
        cnt_delta = my_cnt - self._prev_my_cnt
        if cnt_delta > 0: reward += cnt_delta * 10.0  # captured
        if cnt_delta < 0: reward += cnt_delta * 15.0  # lost (negative)

        # 3. Ship dominance ratio (per tick)
        ship_ratio = my_ships / all_ships
        reward += (ship_ratio - 0.25) * 0.5  # bonus for >25% of total ships

        # 4. Efficiency: reward for expanding early
        if new_stp < 100 and prod_delta > 0:
            reward += prod_delta * 0.5  # double bonus in early game

        # 5. Penalty for idle ships (ships not growing = wasted turns)
        if my_ships < self._prev_ships and cnt_delta >= 0:
            reward -= 0.02  # small penalty for losing ships without gaining planets

        self._prev_my_prod = my_prod
        self._prev_my_cnt  = my_cnt
        self._prev_ships   = my_ships

        # ── Terminal reward ───────────────────────────────────────────
        status = self._env.state[self.player_id].status
        done   = (status != 'ACTIVE')

        if done:
            r = self._env.state[self.player_id].reward
            if   r ==  1: reward += 100.0  # WIN
            elif r == -1: reward -= 50.0   # LOSS
            else:         reward -= 10.0   # DRAW

        truncated = (new_stp >= MS)
        return obs, reward, done, truncated, {}

    def render(self): pass
    def close(self):
        if self._env: self._env = None

# ─── CELL 6: SELF-PLAY TRAINING ──────────────────────────────────────────────
def train_ppo(total_timesteps=500_000, save_path='kronos_ppo'):
    """
    Train PPO agent with reward-shaped OrbitWarsEnv.
    Self-play: agent trains against KRONOS heuristic.
    """
    try:
        from stable_baselines3 import PPO
        from stable_baselines3.common.callbacks import (
            EvalCallback, CheckpointCallback
        )
        from stable_baselines3.common.monitor import Monitor
        import torch

        # Create environments
        train_env = Monitor(OrbitWarsEnv(player_id=0))

        # PPO configuration
        model = PPO(
            'MlpPolicy',
            train_env,
            verbose=1,
            learning_rate=3e-4,
            n_steps=256,
            batch_size=64,
            n_epochs=8,
            gamma=0.995,        # values long-term production
            gae_lambda=0.95,
            clip_range=0.2,
            ent_coef=0.01,      # encourages exploration
            policy_kwargs=dict(
                net_arch=dict(
                    pi=[256, 256, 128],  # policy network
                    vf=[256, 256, 128]   # value network
                ),
                activation_fn=torch.nn.ReLU
            )
        )

        print("🤖 PPO Model architecture:")
        print(f"   Obs dims : {train_env.observation_space.shape}")
        print(f"   Act dims : {train_env.action_space.nvec}")
        print(f"   Network  : [256, 256, 128]")
        print(f"   Params   : {sum(p.numel() for p in model.policy.parameters()):,}")
        print(f"\n🚀 Training for {total_timesteps:,} steps...")

        # Checkpoint every 50k steps
        checkpoint_cb = CheckpointCallback(
            save_freq=50_000,
            save_path='./',
            name_prefix='kronos_checkpoint'
        )

        model.learn(
            total_timesteps=total_timesteps,
            callback=checkpoint_cb,
            progress_bar=True
        )

        model.save(save_path)
        print(f"\n✅ Model saved: {save_path}.zip")
        return model

    except ImportError as e:
        print(f"⚠️  {e}")
        print("   Run: !pip install stable-baselines3 torch")
        return None

# ─── CELL 7: PPO SUBMISSION AGENT ────────────────────────────────────────────
def make_ppo_agent(model_path='kronos_ppo'):
    """
    Load trained PPO model and return submission agent function.
    Falls back to KRONOS heuristic if model not found.
    """
    try:
        from stable_baselines3 import PPO
        model = PPO.load(model_path)
        print(f"✅ PPO model loaded from {model_path}")

        def ppo_orbital_strategist(obs):
            if isinstance(obs,dict):
                pl=obs.get('player',0); rp=obs.get('planets',[])
                rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
                stp=obs.get('step',0)
            else:
                pl=obs.player; rp=obs.planets; rf=obs.fleets
                av=obs.angular_velocity; stp=getattr(obs,'step',0)

            try:
                from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
                planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
            except:
                planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

            mine  =[p for p in planets if p.owner==pl]
            others=[p for p in planets if p.owner!=pl]

            # Build observation
            obs_arr = np.zeros((20,3), dtype=np.float32)
            for i,p in enumerate(planets[:20]):
                obs_arr[i,0] = 1.0 if p.owner==pl else (0.0 if p.owner<0 else -1.0)
                obs_arr[i,1] = min(1.0,p.ships/200.0)
                obs_arr[i,2] = p.production/10.0
            obs_flat = obs_arr.flatten().reshape(1,-1)

            # Predict action
            action,_ = model.predict(obs_flat, deterministic=True)
            src_idx,tgt_idx,frac_idx = int(action[0]),int(action[1]),int(action[2])

            if src_idx>=len(mine) or tgt_idx>=len(others): return []
            src=mine[src_idx]; tgt=others[tgt_idx]
            fraction=(frac_idx+1)/10.0
            n=max(1,int(src.ships*fraction)); n=min(n,src.ships-1)
            if n<=0: return []
            a,dd,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            return [[src.id,sa,n]] if ok else []

        return ppo_orbital_strategist

    except Exception as e:
        print(f"⚠️  PPO model not found ({e}), using KRONOS heuristic")
        return kronos_agent

# ─── SUBMISSION AGENT ────────────────────────────────────────────────────────
# Before training: uses KRONOS heuristic
# After training:  use make_ppo_agent() for PPO inference
orbital_strategist = kronos_agent
agent = orbital_strategist

# After training, uncomment:
# agent = make_ppo_agent('kronos_ppo')


## ✅ Cell 11 — Verify Submission


In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('main','main.py')
mod=importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
sub=mod.agent
print(f'✅ Submission agent: {sub.__name__}')
ev=make('orbit_wars',debug=False)
ev.run([sub,v1_agent,'random',v1_agent])
fr=[s.reward for s in ev.steps[-1]]
print(f'Rewards: {fr}')
print('🏆 WINS!' if fr[0]==1 else '✅ Runs correctly')
